# Follicle counting

This notebook uses a helper script with DBSCAN to:

* Count putative individual B cell follicles in selected, annotated BANKSY domains
* Plot results in barplots and boxplots
* Save a summary of follicle counts across samples

Imports and setup:

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import DBSCAN
from pathlib import Path
import os
import hdbscan

### Define base names for samples of interest

In [ ]:
basenames = [
    "IHOPE14_MedLN_BottomLeft",
    "IHOPE14_MedLN_BottomRight",
    "IHOPE14_MedLN_TopRight",
    "IHOPE14_mesLN",
    "IHOPE20_LN",
    "IHOPE20_Spleen",
    "IHOPE26_LN",
    "IHOPE26_Spleen",
    "IHOPE27_LN",
    "IHOPE27_Spleen",
    "IHOPE39_LN",
    "IHOPE39_MesLN_1",
    "IHOPE39_Spleen"
]

DATA_DIR = Path("../data/processed/anndata/NEW")
OUT_DIR = Path("../results/reports/follicle_counts")
OUT_DIR.mkdir(exist_ok=True, parents=True)

### Apply loop to all samples of interest

In [ ]:
import scripts.follicle_counting_helpers as fh

results = []

for basename in basenames:
    try:
        print(f"\nProcessing {basename}")

        filepath = (
            DATA_DIR /
            f"{basename}_celltypes_follicledomains.h5ad"
        )

        adata = sc.read_h5ad(filepath)

        mask = (
            (adata.obs["type_B"] == True) &
            (adata.obs["B_follicle"] == True)
        )

        print(f"{basename}: {mask.sum()} B cells in follicle regions")

        adata, n_follicles = fh.detect_follicles(adata)

        fh.plot_follicles(adata, basename, OUT_DIR)

        print(f"Plotted {basename}, saved as {OUT_DIR / f'{basename}_follicles.png'}")

        adata.write(
            OUT_DIR / f"{basename}_follicle_clustered.h5ad"
        )

        results.append({
            "sample": basename,
            "n_follicles": n_follicles,
            "n_candidate_Bcells": mask.sum(),
            "total_cells": adata.n_obs
        })

    except Exception as e:
        print(f"Failed on {basename}: {e}")

In [ ]:
results_df = pd.DataFrame(results)

results_df = fh.normalize_follicle_counts(results_df)

results_df.to_csv(
    OUT_DIR / "follicle_counts_summary.csv",
    index=False
)

results_df

**Follicle counts per sample (ranked) – barplot**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 6))

df_sorted = results_df.sort_values("n_follicles", ascending=False)

sns.barplot(
    data=df_sorted,
    x="sample",
    y="n_follicles",
    color="steelblue"
)

plt.xticks(rotation=60, ha="right")
plt.ylabel("Number of follicles")
plt.xlabel("Sample")
plt.title("Follicle counts per sample (ranked)")
plt.tight_layout()
plt.grid(False)


plt.savefig(
    OUT_DIR / "follicle_counts_ranked_barplot.png",
    dpi=300
)

plt.show()

**Normalized**

In [ ]:
plt.figure(figsize=(12, 6))

df_sorted_norm = results_df.sort_values("follicles_per_1000cells", ascending=False)

sns.barplot(
    data=df_sorted_norm,
    x="sample",
    y="follicles_per_1000cells",
    color="steelblue"
)

plt.xticks(rotation=60, ha="right")
plt.ylabel("Follicles per 1000 cells")
plt.xlabel("Sample")
plt.title("Normalized follicle counts per sample (ranked)")
plt.tight_layout()
plt.grid(False)

plt.savefig(
    OUT_DIR / "follicle_counts_ranked_barplot_normalized.png",
    dpi=300
)

plt.show()

**Follicle counts per sample (custom order) – barplot**


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 6))

# Define order manually
custom_order = [
    "IHOPE14_MedLN_BottomLeft",
    "IHOPE14_MedLN_BottomRight",
    "IHOPE14_MedLN_TopRight",
    "IHOPE20_LN",
    "IHOPE26_LN",
    "IHOPE27_LN",
    "IHOPE39_LN",
    "IHOPE14_mesLN",
    "IHOPE39_MesLN_1",
    "IHOPE20_Spleen",
    "IHOPE26_Spleen",
    "IHOPE27_Spleen",
    "IHOPE39_Spleen"
]

sns.barplot(
    data=results_df,
    x="sample",
    y="n_follicles",
    order=custom_order,
    color="steelblue"
)

plt.xticks(rotation=60, ha="right")
plt.ylabel("Number of follicles")
plt.xlabel("Sample")
plt.title("Follicle counts per sample")
plt.tight_layout()
plt.grid(False)

plt.savefig(
    OUT_DIR / "follicle_counts_alphabetical_order_barplot.png",
    dpi=300
)

plt.show()

**Normalized**

In [ ]:
plt.figure(figsize=(12, 6))

sns.barplot(
    data=results_df,
    x="sample",
    y="follicles_per_1000cells",
    order=custom_order,
    color="steelblue"
)

plt.xticks(rotation=60, ha="right")
plt.ylabel("Follicles per 1000 cells")
plt.xlabel("Sample")
plt.title("Normalized follicle counts per sample")
plt.tight_layout()
plt.grid(False)

plt.savefig(
    OUT_DIR / "follicle_counts_alphabetical_order_barplot_normalized.png",
    dpi=300
)

plt.show()

**Follicle counts per sample (grouped by tissue manually) – boxplot**


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

tissue_map = {
    # MedLN
    "IHOPE14_MedLN_BottomLeft": "MedLN",
    "IHOPE14_MedLN_BottomRight": "MedLN",
    "IHOPE14_MedLN_TopRight": "MedLN",
    "IHOPE20_LN": "MedLN",
    "IHOPE26_LN": "MedLN",
    "IHOPE27_LN": "MedLN",
    "IHOPE39_LN": "MedLN",

    # MesLN
    "IHOPE14_mesLN": "MesLN",
    "IHOPE39_MesLN_1": "MesLN",

    # Spleen
    "IHOPE20_Spleen": "Spleen",
    "IHOPE26_Spleen": "Spleen",
    "IHOPE27_Spleen": "Spleen",
    "IHOPE39_Spleen": "Spleen"
}

results_df["tissue"] = results_df["sample"].map(tissue_map)

In [ ]:
plt.figure(figsize=(8,6))

sns.boxplot(
    data=results_df,
    x="tissue",
    y="n_follicles",
    order=["MedLN", "MesLN", "Spleen"]
)

sns.stripplot(
    data=results_df,
    x="tissue",
    y="n_follicles",
    order=["MedLN", "MesLN", "Spleen"],
    color="black",
    alpha=0.7
)

plt.ylabel("Number of follicles")
plt.title("Follicle counts by tissue")
plt.xlabel(None)
plt.tight_layout()
plt.grid(False)

plt.savefig(
    OUT_DIR / "follicle_counts_boxplot_tissue.png",
    dpi=300
)

plt.show()

**Normalized**

In [ ]:
plt.figure(figsize=(8,6))

sns.boxplot(
    data=results_df,
    x="tissue",
    y="follicles_per_1000cells",
    order=["MedLN", "MesLN", "Spleen"]
)

sns.stripplot(
    data=results_df,
    x="tissue",
    y="follicles_per_1000cells",
    order=["MedLN", "MesLN", "Spleen"],
    color="black",
    alpha=0.7
)

plt.ylabel("Follicles per 1000 cells")
plt.title("Normalized follicle counts by tissue")
plt.xlabel(None)
plt.tight_layout()
plt.grid(False)

plt.savefig(
    OUT_DIR / "follicle_counts_boxplot_tissue_normalized.png",
    dpi=300
)

plt.show()

# Summary

Generate a full summary based on the cell type summary ("type" level) and the follicle counts.

In [ ]:
follicle_df = pd.read_csv(
    OUT_DIR / "follicle_counts_summary.csv"
)

follicle_df.head()

In [ ]:
from pathlib import Path
import pandas as pd

summary_dir = Path("../results/reports/NEW")

summary_files = list(
    summary_dir.glob("celltype_summary_*debugged_summary.csv")
)

summary_files

In [ ]:
all_rows = []

for file in summary_files:

    df = pd.read_csv(file)

    sample_name = (
        file.stem
        .replace("celltype_summary_", "")
        .replace("_debugged_summary", "")
    )

    type_df = df[df["level"] == "type"]

    row = {"sample": sample_name}

    if len(type_df) > 0:
        row["total_cells"] = type_df["total_cells"].iloc[0]

    for _, r in type_df.iterrows():
        ct = r["cell_type"]

        row[f"{ct}_count"] = r["n_cells"]
        row[f"{ct}_pct"] = r["pct_total"]

    all_rows.append(row)

celltype_wide = pd.DataFrame(all_rows)

celltype_wide.head()

In [ ]:
final_report = follicle_df.merge(
    celltype_wide,
    on="sample",
    how="left"
)

final_report

In [ ]:
final_report.to_csv(
    OUT_DIR / "follicle_celltype_report.csv",
    index=False
)

final_report